In [ ]:
import cv2
import numpy as np
import time
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
from acquire_zarr import (
    StreamSettings, ZarrStream, Dimension, DimensionType, ZarrVersion,
    DataType, Compressor, CompressionCodec, CompressionSettings
)

In [ ]:
def configure_zarr_stream(frames_to_capture=100, width=640, height=480):
    """Configure the Zarr stream settings for webcam data."""
    settings = StreamSettings()

    # Configure compression (using LZ4 as in the example)
    settings.compression = CompressionSettings(
        compressor=Compressor.BLOSC1,
        codec=CompressionCodec.BLOSC_LZ4,
        level=1,
        shuffle=1,
    )

    # We'll use a 4D array structure:
    # - t: time dimension (frame number)
    # - c: channel (3 for RGB)
    # - y: height
    # - x: width
    settings.dimensions.extend([
        Dimension(
            name="t",
            kind=DimensionType.TIME,
            array_size_px=frames_to_capture,  # Total frames we plan to capture
            chunk_size_px=10,  # Store frames in chunks of 10
            shard_size_chunks=1,
        ),
        Dimension(
            name="c",
            kind=DimensionType.CHANNEL,
            array_size_px=3,  # RGB channels
            chunk_size_px=3,  # Keep channels together
            shard_size_chunks=1,
        ),
        Dimension(
            name="y",
            kind=DimensionType.SPACE,
            array_size_px=height,
            chunk_size_px=height,  # Keep full height in one chunk
            shard_size_chunks=1,
        ),
        Dimension(
            name="x",
            kind=DimensionType.SPACE,
            array_size_px=width,
            chunk_size_px=width,  # Keep full width in one chunk
            shard_size_chunks=1,
        ),
    ])

    settings.store_path = "webcam_capture.zarr"
    settings.version = ZarrVersion.V3
    settings.data_type = DataType.UINT8  # Camera data typically comes as uint8

    return settings

In [ ]:
def initialize_webcam(cam_id=0):
    """Initialize the webcam and return the capture object."""
    cap = cv2.VideoCapture(cam_id)
    assert cap.isOpened(), "Could not open webcam"
    return cap

In [ ]:
def display_frame(frame):
    """Display the current frame for monitoring."""
    # Convert from BGR (OpenCV format) to RGB (for matplotlib)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(rgb_frame)
    plt.axis('off')
    plt.title('Webcam Capture')
    display(plt.gcf())
    clear_output(wait=True)
    plt.close()

In [ ]:
def capture_and_stream(frames_to_capture=100, preview_interval=10):
    """Capture frames from webcam and stream to Zarr."""
    # Initialize webcam
    cap = initialize_webcam()

    # Get actual webcam resolution
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Webcam resolution: {width}x{height}")

    # Configure and create zarr stream
    settings = configure_zarr_stream(frames_to_capture, width, height)
    stream = ZarrStream(settings)

    # Prepare a buffer to hold multiple frames before writing
    buffer_size = 10  # Number of frames to buffer before writing
    frame_buffer = np.zeros((buffer_size, 3, height, width), dtype=np.uint8)

    # Capture loop
    try:
        frame_count = 0
        buffer_count = 0
        start_time = time.time()

        while frame_count < frames_to_capture:
            ret, frame = cap.read()
            if not ret:
                print("Failed to capture frame")
                break

            # Display preview occasionally
            if frame_count % preview_interval == 0:
                display_frame(frame)

            # Convert from BGR (OpenCV) to RGB and transpose to (c, y, x)
            # OpenCV gives (y, x, c) but we want (c, y, x) for our Zarr dimensions
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_transposed = np.transpose(frame_rgb, (2, 0, 1))

            # Add to buffer
            buffer_index = buffer_count % buffer_size
            frame_buffer[buffer_index] = frame_transposed
            buffer_count += 1

            # When buffer is full, append to Zarr stream
            if buffer_count % buffer_size == 0:
                # Reshape buffer to match expected dimensions (t, c, y, x)
                data_chunk = frame_buffer.copy()
                stream.append(data_chunk)
                print(f"Appended frames {frame_count - buffer_size + 1} to {frame_count}")

            frame_count += 1

        # Write any remaining frames in buffer
        remaining = buffer_count % buffer_size
        if remaining > 0:
            stream.append(frame_buffer[:remaining])
            print(f"Appended final {remaining} frames")

    finally:
        cap.release()
        elapsed = time.time() - start_time
        print(f"Captured {frame_count} frames in {elapsed:.2f} seconds ({frame_count / elapsed:.2f} fps)")

In [ ]:
def read_zarr_data(zarr_path="webcam_capture.zarr", frame_index=0):
    """Read and display a frame from the Zarr store."""
    import zarr

    # Open the Zarr store
    group = zarr.open(zarr_path, mode='r')

    # Get a specific frame
    array = group["0"]
    frame = array[frame_index]

    # If the frame is stored as (c, y, x), transpose back to (y, x, c) for display
    if frame.shape[0] == 3:  # If first dimension is channels
        frame = np.transpose(frame, (1, 2, 0))

    # Display the frame
    plt.figure(figsize=(8, 6))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(f'Frame {frame_index} from Zarr Store')
    plt.show()

    return frame

In [ ]:
capture_and_stream(frames_to_capture=50)

In [ ]:
frame = read_zarr_data(frame_index=49)